# CS3807 Deep Learning Laboratory - Part 3: Model Construction and Training
## Experiments 1 & 2: Attention Type and Position Ablations

In [ ]:
# Cell 1: Load all prerequisites
import os
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf

for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tqdm import tqdm
import matplotlib.pyplot as plt
import pickle

# Load data
with open('results/data_splits.pkl', 'rb') as f:
    splits = pickle.load(f)

X_train, y_train = splits['X_train'], splits['y_train']
X_val, y_val = splits['X_val'], splits['y_val']
X_test, y_test = splits['X_test'], splits['y_test']
class_names = splits['class_names']
num_classes = splits['num_classes']
train_augment = splits['train_augment']

print(f"✓ Data loaded")
print(f"  Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

In [ ]:
# Cell 2: Import attention modules from Part 2 (no code duplication)
with open('results/attention_modules.pkl', 'rb') as f:
    attention_registry = pickle.load(f)

# Import all attention classes
ChannelAttention = attention_registry['ChannelAttention']
SpatialAttention = attention_registry['SpatialAttention']
ChannelSpatialAttention = attention_registry['ChannelSpatialAttention']
SelfAttention = attention_registry['SelfAttention']
SEAttention = attention_registry['SEAttention']
ECAAttention = attention_registry['ECAAttention']
CBAM = attention_registry['CBAM']
CoordinateAttention = attention_registry['CoordinateAttention']
VariableSpatialAttention = attention_registry['VariableSpatialAttention']

print("✓ Attention modules imported from Part 2")
print(f"  Loaded {len(attention_registry)} attention mechanisms")

## SECTION 1: MODEL CONSTRUCTION UTILITIES

In [ ]:
# Cell 4: Test model construction
# Clear GPU memory before building test model
tf.keras.backend.clear_session()

test_model = build_model_with_attention(attention_type='channel', position='P3')
print("\nTest Model Summary:")
test_model.summary()
print(f"\n✓ Model built successfully")

# Clean up test model to free memory
del test_model
tf.keras.backend.clear_session()
print("✓ Test model cleaned up")

# Cell 6: Define Experiment 1 models (Attention Type Ablation at P3)
exp1_models = {
    'A0_Baseline': {'type': 'none', 'position': 'P3'},
    'A1_Channel': {'type': 'channel', 'position': 'P3'},
    'A2_Spatial': {'type': 'spatial', 'position': 'P3'},
    'A3_ChannelSpatial': {'type': 'channel_spatial', 'position': 'P3'},
    'A4_Self': {'type': 'self', 'position': 'P3'},
    'A5_Combined': {'type': 'combined', 'position': 'P3'}
}

print("Experiment 1: Attention-Type Ablation (Fixed Position P3)")
print("="*60)
for model_name, config in exp1_models.items():
    print(f"  {model_name}: Type={config['type']}, Position={config['position']}")

print(f"\nTotal models to train: {len(exp1_models)}")

In [ ]:
# Cell 7: Training function with early stopping and learning rate scheduling
def train_model(model, X_train, y_train, X_val, y_val, model_name, epochs=30):
    """
    Train model with validation and save results
    """
    
    # Callbacks
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True,
            verbose=0
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=0
        )
    ]
    
    # Train
    print(f"\nTraining {model_name}...")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        batch_size=BATCH_SIZE,
        epochs=epochs,
        callbacks=callbacks,
        verbose=0
    )
    
    # Save model and history
    model.save(f'results/models/{model_name}_model.h5')
    with open(f'results/models/{model_name}_history.pkl', 'wb') as f:
        pickle.dump(history.history, f)
    
    print(f"  ✓ Training completed (Epochs: {len(history.history['loss'])})")
    print(f"    Final train acc: {history.history['accuracy'][-1]:.4f}")
    print(f"    Final val acc: {history.history['val_accuracy'][-1]:.4f}")
    
    return history

print("✓ Training function defined")

In [ ]:
# Cell 8: Train all Experiment 1 models (MAIN TRAINING LOOP)
exp1_histories = {}
exp1_trained_models = {}

for idx, (model_name, config) in enumerate(exp1_models.items(), 1):
    print(f"\n[{idx}/{len(exp1_models)}] Building {model_name}...")
    
    # Clear GPU memory before building new model
    tf.keras.backend.clear_session()
    
    # Build model
    model = build_model_with_attention(
        attention_type=config['type'],
        position=config['position']
    )
    
    # Train model
    history = train_model(
        model, X_train, y_train, X_val, y_val, 
        model_name, epochs=EPOCHS
    )
    
    # Store
    exp1_histories[model_name] = history.history
    exp1_trained_models[model_name] = model
    
    # Clear GPU memory after training
    del model
    tf.keras.backend.clear_session()

print("\n" + "="*60)
print("✓ EXPERIMENT 1 COMPLETE: All models trained")
print("="*60)

In [ ]:
# Cell 5: Define training configuration with GPU memory detection
import tensorflow as tf

# Detect GPU memory and set appropriate batch size
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Get GPU memory info (works on recent TensorFlow versions)
        gpu_memory_gb = 4  # Default fallback
        print(f"GPU detected: {gpus[0].name}")
        
        # Set batch size based on GPU memory (conservative estimate)
        if gpu_memory_gb >= 16:
            BATCH_SIZE = 32
        elif gpu_memory_gb >= 8:
            BATCH_SIZE = 16
        elif gpu_memory_gb >= 6:
            BATCH_SIZE = 8
        else:
            BATCH_SIZE = 4  # For 4GB GPUs
    except:
        # Fallback if memory detection fails
        BATCH_SIZE = 16  # Safe default
        print("Could not detect GPU memory, using default batch_size=16")
else:
    BATCH_SIZE = 8  # CPU default
    print("No GPU detected, using CPU with batch_size=8")

EPOCHS = 30
INITIAL_LR = 1e-3
FINETUNE_LR = 1e-4
VALIDATION_SPLIT = 0.2

training_config = {
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'initial_lr': INITIAL_LR,
    'finetune_lr': FINETUNE_LR,
    'validation_split': VALIDATION_SPLIT,
    'optimizer': 'Adam',
    'loss': 'CategoricalCrossentropy'
}

print("Training Configuration (IDENTICAL FOR ALL MODELS):")
print("="*50)
for key, val in training_config.items():
    print(f"  {key}: {val}")

# Save config
with open('results/training_config.pkl', 'wb') as f:
    pickle.dump(training_config, f)

In [ ]:
# Cell 10: Define Experiment 2 models (Attention Position Ablation)
exp2_models = {
    'B2_P2': {'type': 'channel_spatial', 'position': 'P2'},
    'B3_P3': {'type': 'channel_spatial', 'position': 'P3'},
    'B4_P4': {'type': 'channel_spatial', 'position': 'P4'}
}

print("Experiment 2: Attention-Position Ablation (Fixed Type: Channel-Spatial)")
print("="*70)
for model_name, config in exp2_models.items():
    print(f"  {model_name}: Type={config['type']}, Position={config['position']}")

print(f"\nTotal models to train: {len(exp2_models)}")
print("Note: B1 (P1) skipped due to high spatial resolution (computational cost)")

# Cell 11: Train all Experiment 2 models
exp2_histories = {}
exp2_trained_models = {}

for idx, (model_name, config) in enumerate(exp2_models.items(), 1):
    print(f"\n[{idx}/{len(exp2_models)}] Building {model_name}...")
    
    # Clear GPU memory
    tf.keras.backend.clear_session()
    
    # Build model
    model = build_model_with_attention(
        attention_type=config['type'],
        position=config['position']
    )
    
    # Train model
    history = train_model(
        model, X_train, y_train, X_val, y_val, 
        model_name, epochs=EPOCHS
    )
    
    # Store
    exp2_histories[model_name] = history.history
    exp2_trained_models[model_name] = model
    
    # Clear GPU memory
    del model
    tf.keras.backend.clear_session()

print("\n" + "="*60)
print("✓ EXPERIMENT 2 COMPLETE: All models trained")
print("="*60)

In [ ]:
# Cell 12: Save Experiment 2 histories
with open('results/exp2_histories.pkl', 'wb') as f:
    pickle.dump(exp2_histories, f)

print("✓ Experiment 2 histories saved")

In [ ]:
# Cell 13: Define Experiment 3 models (multi-position insertion)
exp3_models = {
    'C1_P2P3': {'type': 'channel_spatial', 'positions': ('P2', 'P3')},
    'C2_P3P4': {'type': 'channel_spatial', 'positions': ('P3', 'P4')},
    'C3_P2P3P4': {'type': 'channel_spatial', 'positions': ('P2', 'P3', 'P4')},
}

print("Experiment 3: Multiple Attention Positions")
print("="*60)
for model_name, config in exp3_models.items():
    print(f"  {model_name}: Type={config['type']}, Positions={config['positions']}")

print(f"\nTotal models to train: {len(exp3_models)}")

In [ ]:
# Cell 14: Train all Experiment 3 models
exp3_histories = {}
exp3_trained_models = {}

for idx, (model_name, config) in enumerate(exp3_models.items(), 1):
    print(f"\n[{idx}/{len(exp3_models)}] Building {model_name}...")
    
    # Clear GPU memory
    tf.keras.backend.clear_session()
    
    # Build model (use multi-position function)
    model = build_model_multi_position(
        attention_type=config['type'],
        positions=config['positions']
    )
    
    # Train model
    history = train_model(
        model, X_train, y_train, X_val, y_val, 
        model_name, epochs=EPOCHS
    )
    
    # Store
    exp3_histories[model_name] = history.history
    exp3_trained_models[model_name] = model
    
    # Clear GPU memory
    del model
    tf.keras.backend.clear_session()

print("\n" + "="*60)
print("✓ EXPERIMENT 3 COMPLETE: All models trained")
print("="*60)

# Cell 15: Save Experiment 3 histories
with open('results/exp3_histories.pkl', 'wb') as f:
    pickle.dump(exp3_histories, f)

print("✓ Experiment 3 histories saved")

In [ ]:
# Cell 16: Define Experiment 4 models (Attention Order Ablation)
exp4_models = {
    'D1_ChannelSpatial': {'type': 'channel_spatial', 'position': 'P3'},
    'D2_SpatialChannel': {'type': 'spatial_channel', 'position': 'P3'}
}

print("Experiment 4: Attention Order Ablation (Fixed Position P3)")
print("="*60)
for model_name, config in exp4_models.items():
    print(f"  {model_name}: Type={config['type']}, Position={config['position']}")

print(f"\nTotal models to train: {len(exp4_models)}")

In [ ]:
# Cell 17: Train all Experiment 4 models
exp4_histories = {}
exp4_trained_models = {}

for idx, (model_name, config) in enumerate(exp4_models.items(), 1):
    print(f"\n[{idx}/{len(exp4_models)}] Building {model_name}...")
    
    # Clear GPU memory
    tf.keras.backend.clear_session()
    
    # Build model
    model = build_model_with_attention(
        attention_type=config['type'],
        position=config['position']
    )
    
    # Train model
    history = train_model(
        model, X_train, y_train, X_val, y_val, 
        model_name, epochs=EPOCHS
    )
    
    # Store
    exp4_histories[model_name] = history.history
    exp4_trained_models[model_name] = model
    
    # Clear GPU memory
    del model
    tf.keras.backend.clear_session()

print("\n" + "="*60)
print("✓ EXPERIMENT 4 COMPLETE: All models trained")
print("="*60)

# Cell 18: Save Experiment 4 histories
with open('results/exp4_histories.pkl', 'wb') as f:
    pickle.dump(exp4_histories, f)

print("✓ Experiment 4 histories saved")

In [ ]:
# Cell 19: Define Experiment 5 models (Spatial Kernel Size Ablation)
exp5_models = {
    'E1_Kernel3': {'type': 'spatial', 'position': 'P3', 'kernel': 3},
    'E2_Kernel5': {'type': 'spatial', 'position': 'P3', 'kernel': 5},
    'E3_Kernel7': {'type': 'spatial', 'position': 'P3', 'kernel': 7}
}

print("Experiment 5: Spatial Kernel Size Ablation (Fixed Position P3)")
print("="*60)
for model_name, config in exp5_models.items():
    print(f"  {model_name}: Type={config['type']}, Position={config['position']}, Kernel={config['kernel']}")

print(f"\nTotal models to train: {len(exp5_models)}")

# Cell 20: Train all Experiment 5 models
exp5_histories = {}
exp5_trained_models = {}

for idx, (model_name, config) in enumerate(exp5_models.items(), 1):
    print(f"\n[{idx}/{len(exp5_models)}] Building {model_name}...")
    
    # Clear GPU memory
    tf.keras.backend.clear_session()
    
    # Build model with custom kernel size
    model = build_model_with_attention(
        attention_type=config['type'],
        position=config['position'],
        spatial_kernel=config['kernel']
    )
    
    # Train model
    history = train_model(
        model, X_train, y_train, X_val, y_val, 
        model_name, epochs=EPOCHS
    )
    
    # Store
    exp5_histories[model_name] = history.history
    exp5_trained_models[model_name] = model
    
    # Clear GPU memory
    del model
    tf.keras.backend.clear_session()

print("\n" + "="*60)
print("✓ EXPERIMENT 5 COMPLETE: All models trained")
print("="*60)

In [ ]:
# Cell 21: Save Experiment 5 histories
with open('results/exp5_histories.pkl', 'wb') as f:
    pickle.dump(exp5_histories, f)

print("✓ Experiment 5 histories saved")

# Cell 22: Define Additional models (SE, ECA, CBAM, Coordinate)
additional_models = {
    'F1_SE': {'type': 'se', 'position': 'P3'},
    'F2_ECA': {'type': 'eca', 'position': 'P3'},
    'F3_CBAM': {'type': 'cbam', 'position': 'P3'},
    'F4_Coordinate': {'type': 'coordinate', 'position': 'P3'}
}

print("Additional Experiments: Extended Attention Mechanisms")
print("="*60)
for model_name, config in additional_models.items():
    print(f"  {model_name}: Type={config['type']}, Position={config['position']}")

print(f"\nTotal models to train: {len(additional_models)}")

In [ ]:
# Cell 23: Train all Additional models
additional_histories = {}
additional_trained_models = {}

for idx, (model_name, config) in enumerate(additional_models.items(), 1):
    print(f"\n[{idx}/{len(additional_models)}] Building {model_name}...")
    
    # Clear GPU memory
    tf.keras.backend.clear_session()
    
    # Build model
    model = build_model_with_attention(
        attention_type=config['type'],
        position=config['position']
    )
    
    # Train model
    history = train_model(
        model, X_train, y_train, X_val, y_val, 
        model_name, epochs=EPOCHS
    )
    
    # Store
    additional_histories[model_name] = history.history
    additional_trained_models[model_name] = model
    
    # Clear GPU memory
    del model
    tf.keras.backend.clear_session()

print("\n" + "="*60)
print("✓ ADDITIONAL EXPERIMENTS COMPLETE: All models trained")
print("="*60)

# Cell 24: Save Additional histories
with open('results/additional_histories.pkl', 'wb') as f:
    pickle.dump(additional_histories, f)

print("✓ Additional histories saved")